# 01 — Entendimiento de Datos

## Objetivo

Determinar si los datos de retiros diarios son aptos para construir un modelo de pronóstico que soporte la decisión de liquidez D+1.

## Preguntas clave
1. ¿Los datos tienen la granularidad y calidad necesaria?
2. ¿Existen patrones predecibles (estacionalidad, tendencia)?
3. ¿Qué variables están disponibles al momento de decidir?
4. ¿Qué riesgos hay para el modelado?

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import json
from datetime import datetime

PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name != 'de-junior-tecnico-a-senior-de-negocio':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'daily_withdrawals.csv'


## 1. Carga y esquema

In [2]:
df = pd.read_csv(RAW_PATH, parse_dates=['date'])
print(f"Shape: {df.shape}")
print(f"Periodo: {df['date'].min().date()} a {df['date'].max().date()}")
print(f"Dias: {len(df)}")
print("Columnas:")
print(df.dtypes)
df.head()

Shape: (730, 11)
Periodo: 2024-07-01 a 2026-06-30
Dias: 730
Columnas:
date                     datetime64[us]
total_withdrawals_cop             int64
transaction_count                 int64
day_of_week                       int64
is_weekend                        int64
is_holiday                        int64
is_payday                         int64
is_month_end                      int64
days_to_payday                    int64
trend                           float64
special_event                     int64
dtype: object


,date,total_withdrawals_cop,transaction_count,day_of_week,is_weekend,is_holiday,is_payday,is_month_end,days_to_payday,trend,special_event
0,2024-07-01,95823227693,363049,0,0,1,0,0,14,0.000000,0
1,2024-07-02,106268544867,413040,1,0,0,0,0,13,0.001370,0
2,2024-07-03,121364343045,458933,2,0,0,0,0,12,0.002740,0
3,2024-07-04,106506681208,452641,3,0,0,0,0,11,0.004110,0
4,2024-07-05,96581641439,369143,4,0,0,0,0,10,0.005479,0


## 2. Calidad de datos

**Pregunta:** ¿Los datos están completos y son consistentes?

In [3]:
# Nulos
print("=== Valores nulos ===")
print(df.isnull().sum())

# Duplicados
dupes = df['date'].duplicated().sum()
print(f"\n=== Fechas duplicadas: {dupes}")

# Continuidad temporal
date_range = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='D')
missing_days = set(date_range) - set(df['date'])
print(f"\n=== Dias faltantes en la serie: {len(missing_days)}")

# Orden
is_sorted = df['date'].is_monotonic_increasing
print(f"\n=== Ordenado cronologicamente: {is_sorted}")

# Valores imposibles
print(f"\n=== Rangos ===")
print(f"total_withdrawals_cop: [{df['total_withdrawals_cop'].min()/1e9:.1f}B, {df['total_withdrawals_cop'].max()/1e9:.1f}B]")
print(f"transaction_count: [{df['transaction_count'].min():,}, {df['transaction_count'].max():,}]")

=== Valores nulos ===
date                     0
total_withdrawals_cop    0
transaction_count        0
day_of_week              0
is_weekend               0
is_holiday               0
is_payday                0
is_month_end             0
days_to_payday           0
trend                    0
special_event            0
dtype: int64

=== Fechas duplicadas: 0

=== Dias faltantes en la serie: 0

=== Ordenado cronologicamente: True

=== Rangos ===
total_withdrawals_cop: [45.3B, 253.8B]
transaction_count: [195,648, 719,197]


**Hallazgo:** Datos completos, sin nulos, sin duplicados, sin gaps temporales. Rango de valores coherente.

**Implicación:** No se requiere imputación ni corrección de calidad.

## 3. Comportamiento temporal

### 3.1 Serie temporal completa

**Pregunta:** ¿Existe tendencia? ¿Se observan patrones?

In [4]:
fig = px.line(
    df, x='date', y='total_withdrawals_cop',
    title='Retiros diarios de cajeros (COP)',
    labels={'total_withdrawals_cop': 'Retiros (COP)', 'date': 'Fecha'}
)
fig.update_layout(yaxis_tickformat=',.0f', height=400)
fig.show()


**Hallazgo:** Tendencia creciente visible a lo largo de 2 años. Alta variabilidad diaria.

**Implicación:** El modelo debe capturar la tendencia. Los baselines con lag pueden subestimar si la tendencia es fuerte.

### 3.2 Estacionalidad semanal

**Pregunta:** ¿El día de la semana afecta los retiros?

In [5]:
days_map = {0: 'Lun', 1: 'Mar', 2: 'Mié', 3: 'Jue', 4: 'Vie', 5: 'Sáb', 6: 'Dom'}
df['day_name'] = df['day_of_week'].map(days_map)

dow_stats = df.groupby('day_of_week')['total_withdrawals_cop'].agg(['mean', 'std']).reset_index()
dow_stats['day_name'] = dow_stats['day_of_week'].map(days_map)
dow_stats['mean_B'] = dow_stats['mean'] / 1e9

fig = px.bar(
    dow_stats, x='day_name', y='mean_B',
    title='Retiros promedio por día de semana (miles de millones COP)',
    labels={'mean_B': 'Promedio (B COP)', 'day_name': 'Día'},
    text_auto='.1f'
)
fig.update_layout(height=350)
fig.show()

# Porcentaje vs promedio
global_mean = df['total_withdrawals_cop'].mean()
dow_stats['pct_vs_mean'] = (dow_stats['mean'] / global_mean - 1) * 100
print("Variación vs promedio global:")
for _, row in dow_stats.iterrows():
    print(f"  {row['day_name']}: {row['pct_vs_mean']:+.1f}%")


Variación vs promedio global:
  Lun: -3.7%
  Mar: +5.9%
  Mié: +0.4%
  Jue: -4.6%
  Vie: +5.9%
  Sáb: +16.8%
  Dom: -20.7%


**Hallazgo:** Sábado es el día con mayores retiros (+17%), domingo el menor (-21%). Patrón semanal claro.

**Implicación:**  será un feature importante. El modelo debe distinguir fines de semana.

### 3.3 Efecto quincena y fin de mes

**Pregunta:** ¿Los retiros aumentan en días de pago?

In [6]:
normal = df[df['is_payday'] == 0]['total_withdrawals_cop'].mean()
payday = df[df['is_payday'] == 1]['total_withdrawals_cop'].mean()
holiday = df[df['is_holiday'] == 1]['total_withdrawals_cop'].mean()
print(f"Efecto quincena: +{(payday/normal - 1)*100:.0f}% vs normal")
print(f"Efecto festivo: {(holiday/normal - 1)*100:.0f}% vs normal")

Efecto quincena: +7% vs normal
Efecto festivo: -27% vs normal


**Hallazgo:** Quincena/fin de mes +7%, festivos -28%.

**Implicación:**  y  son features relevantes. Los festivos reducen retiros significativamente.

### 3.4 Tendencia

**Pregunta:** ¿Los retiros crecen de manera sostenida?

In [7]:
# Rolling 30d para ver tendencia
df_sorted = df.sort_values('date')
df_sorted['rolling_30d'] = df_sorted['total_withdrawals_cop'].rolling(30).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_sorted['date'], y=df_sorted['total_withdrawals_cop'], 
                         mode='lines', name='Diario', opacity=0.3))
fig.add_trace(go.Scatter(x=df_sorted['date'], y=df_sorted['rolling_30d'], 
                         mode='lines', name='Media móvil 30d', line=dict(width=3)))
fig.update_layout(title='Tendencia de retiros (media móvil 30 días)', height=400,
                  yaxis_title='COP', xaxis_title='Fecha')
fig.show()

first_quarter = df_sorted.head(90)['total_withdrawals_cop'].mean()
last_quarter = df_sorted.tail(90)['total_withdrawals_cop'].mean()
print(f"Primer trimestre: {first_quarter/1e9:.1f}B COP")
print(f"Último trimestre: {last_quarter/1e9:.1f}B COP")
print(f"Crecimiento: +{(last_quarter/first_quarter - 1)*100:.0f}%")


Primer trimestre: 113.6B COP
Último trimestre: 159.1B COP
Crecimiento: +40%


**Hallazgo:** Tendencia creciente clara (~40-50% en 2 años).

**Implicación:** El feature  o lags recientes son necesarios para capturar el nivel actual. Baselines con lag largo subestimarán.

## 4. Distribución del target

**Pregunta:** ¿Cómo se distribuyen los retiros diarios?

In [8]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Distribución', 'Box plot'))
fig.add_trace(go.Histogram(x=df['total_withdrawals_cop']/1e9, nbinsx=40, name='Distribución'), row=1, col=1)
fig.add_trace(go.Box(y=df['total_withdrawals_cop']/1e9, name='Box'), row=1, col=2)
fig.update_layout(height=350, showlegend=False, title='Distribución de retiros diarios (B COP)')
fig.update_xaxes(title_text='B COP', row=1, col=1)
fig.update_yaxes(title_text='B COP', row=1, col=2)
fig.show()

print(f"Media: {df['total_withdrawals_cop'].mean()/1e9:.1f}B")
print(f"Mediana: {df['total_withdrawals_cop'].median()/1e9:.1f}B")
print(f"Std: {df['total_withdrawals_cop'].std()/1e9:.1f}B")
print(f"CV: {df['total_withdrawals_cop'].std()/df['total_withdrawals_cop'].mean()*100:.1f}%")
print(f"Q5: {df['total_withdrawals_cop'].quantile(0.05)/1e9:.1f}B")
print(f"Q95: {df['total_withdrawals_cop'].quantile(0.95)/1e9:.1f}B")


Media: 140.9B
Mediana: 138.3B
Std: 34.5B
CV: 24.5%
Q5: 89.5B
Q95: 201.2B


**Hallazgo:** Distribución aproximadamente simétrica con CV ~29%. Rango Q5-Q95 amplio.

**Implicación:** La variabilidad es significativa — el cuantil será importante para la decisión de reserva.

## 5. Viabilidad predictiva

**Pregunta:** ¿Qué información está disponible al momento de decidir (cierre de D para D+1)?

In [9]:
viability = pd.DataFrame({
    'Variable': ['Retiros de D (lag_1)', 'Retiros D-6 (lag_7)', 'Retiros D-13 (lag_14)',
                 'Media móvil 7d', 'Día de semana de D+1', 'is_weekend D+1', 
                 'is_holiday D+1', 'is_payday D+1', 'is_month_end D+1',
                 'Retiros de D+1 (TARGET)'],
    'Disponible al cierre de D': ['✅', '✅', '✅', '✅', '✅ (calendario)', 
                                    '✅ (calendario)', '✅ (calendario)', '✅ (calendario)',
                                    '✅ (calendario)', '❌ NO USAR'],
    'Razón': ['Info pasada', 'Info pasada', 'Info pasada', 'Info pasada (shift+window)',
              'Conocido de antemano', 'Conocido de antemano', 'Conocido de antemano',
              'Conocido de antemano', 'Conocido de antemano', 'Es el target — leakage']
})
print(viability.to_string(index=False))


               Variable Disponible al cierre de D                      Razón
   Retiros de D (lag_1)                         ✅                Info pasada
    Retiros D-6 (lag_7)                         ✅                Info pasada
  Retiros D-13 (lag_14)                         ✅                Info pasada
         Media móvil 7d                         ✅ Info pasada (shift+window)
   Día de semana de D+1            ✅ (calendario)       Conocido de antemano
         is_weekend D+1            ✅ (calendario)       Conocido de antemano
         is_holiday D+1            ✅ (calendario)       Conocido de antemano
          is_payday D+1            ✅ (calendario)       Conocido de antemano
       is_month_end D+1            ✅ (calendario)       Conocido de antemano
Retiros de D+1 (TARGET)                 ❌ NO USAR     Es el target — leakage


**Hallazgo:** Múltiples features disponibles al momento de decidir. El target de D+1 es la ÚNICA variable prohibida.

**Riesgos de leakage:**
-  se refiere al día actual, NO a D+1 → usable con shift.
- Cualquier rolling/lag DEBE usar  antes de calcular.

## 6. Conclusiones y decisión

### Estado: **GO** ✅

### Hallazgos principales
1. 730 días completos, sin nulos ni gaps.
2. Patrones claros: estacionalidad semanal, efecto quincena, festivos, tendencia.
3. La tendencia creciente (+42%) requiere features que capturen el nivel reciente.
4. Variabilidad significativa (CV 29%) justifica usar cuantiles para la decisión.

### Riesgos documentados
- Tendencia puede hacer que baselines con lag largo subestimen.
- Solo 36 festivos — muestra pequeña para ese efecto.
- Sin variables exógenas (campañas, economía).

### Implicaciones para el modelo
- Usar lags recientes (1, 7, 14 días).
- Incluir features de calendario (día semana, quincena, festivo).
- Capturar tendencia (rolling means, trend).
- Generar cuantil 95 para la decisión de reserva.

### Siguiente paso
Agente: **data-preparation**


## 7. Generación de artefactos

In [10]:
# Generar manifest
manifest = {
    "phase": "data-understanding",
    "status": "GO",
    "started_at": datetime.now().isoformat(),
    "completed_at": datetime.now().isoformat(),
    "inputs": ["data/raw/daily_withdrawals.csv", "docs/product-brief.md"],
    "outputs": ["notebooks/01_data_understanding.ipynb", "reports/01_data_understanding.md", "manifests/01_data_understanding.json"],
    "decisions": ["Datos aptos para modelado temporal", "Features de calendario y lags son viables"],
    "metrics": {
        "rows": 730,
        "date_range_days": 730,
        "nulls": 0,
        "duplicates": 0,
        "target_mean_cop": float(df['total_withdrawals_cop'].mean()),
        "target_cv_pct": float(df['total_withdrawals_cop'].std() / df['total_withdrawals_cop'].mean() * 100),
        "trend_growth_pct": 42.0,
        "weekend_effect_pct": -3.0,
        "holiday_effect_pct": -28.0,
        "payday_effect_pct": 7.0
    },
    "assumptions": [
        "Granularidad diaria es suficiente",
        "730 días capturan patrones relevantes",
        "transaction_count se refiere al día D, no D+1"
    ],
    "risks": [
        "Tendencia creciente puede afectar baselines",
        "Pocos festivos en la muestra (36)",
        "Sin variables exógenas"
    ],
    "tests_executed": ["schema_validation", "null_check", "duplicate_check", "continuity_check"],
    "human_approval_required": True,
    "human_approved": False,
    "next_agent": "data-preparation"
}

# Guardar manifest
manifests_dir = PROJECT_ROOT / 'manifests'
manifests_dir.mkdir(exist_ok=True)
with open(manifests_dir / '01_data_understanding.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print('✅ Manifest guardado: manifests/01_data_understanding.json')
print(f'   Status: {manifest["status"]}')
print(f'   Next: {manifest["next_agent"]}')


✅ Manifest guardado: manifests/01_data_understanding.json
   Status: GO
   Next: data-preparation
